# Subdivisión de Cuadrantes - Equal Area Strips

Este notebook implementa algoritmos para subdividir cuadrantes en subcuadrantes de área aproximadamente igual usando dos enfoques geométricos:

1. **Equal-Area Strips**: Franjas orientadas según el eje principal del polígono
2. **Recursive Binary Split**: División binaria recursiva

## Objetivo
Generar subcuadrantes con área objetivo de ~0.7 km² en el rango de 0.5-1.1 km².

## [SC-00] Imports y Constantes

Configuración inicial del notebook con todas las librerías necesarias y parámetros por defecto.

In [29]:
import geopandas as gpd
from shapely import ops, affinity
from shapely.ops import split
import numpy as np
import math
import folium
import matplotlib.pyplot as plt  # para debug opcional

# Parámetros por defecto
CUADRANTES_PATH = "../geojson/cuadrantes_cali_rutas_consultores.geojson"
PROJ_CRS = "EPSG:32618"  # métrico
TARGET_MEAN_AREA_KM2 = 0.7
AREA_RANGE_KM2 = (0.5, 1.1)  # min, max aceptables
MIN_WIDTH_M = 120.0
OUTPUT_HTML = "pruebas/subcuadrantes_equal_area.html"

## [SC-01] Lectura y Selección

Función para cargar cuadrantes y seleccionar uno específico por código. Maneja MultiPolygons y calcula el área total.

In [30]:
def load_cuadrante(cuadrantes_path, proj_crs):
    """
    Carga cuadrantes y los reproyecta al CRS especificado
    """
    gdf = gpd.read_file(cuadrantes_path)
    gdf_proj = gdf.to_crs(proj_crs)
    return gdf_proj

# Cargar cuadrantes y seleccionar uno
gdf_cuads = load_cuadrante(CUADRANTES_PATH, PROJ_CRS)

# Seleccionar cuadrante por código
codigo_obj = "CL_3_01"
cuadrante_sel = gdf_cuads[gdf_cuads['codigo'] == codigo_obj]

if len(cuadrante_sel) == 0:
    print(f"No se encontró el cuadrante {codigo_obj}")
else:
    # Obtener geometría y asegurar que sea un polígono único
    geom = cuadrante_sel.geometry.iloc[0]
    
    # Si es MultiPolygon, hacer unary_union
    if geom.geom_type == 'MultiPolygon':
        geom = ops.unary_union(geom)
    
    # Calcular área total en km²
    area_total_m2 = geom.area
    area_total_km2 = area_total_m2 / 1_000_000
    
    print(f"Cuadrante seleccionado: {codigo_obj}")
    print(f"Área total: {area_total_km2:.2f} km²")
    print(f"Geometría: {geom.geom_type}")

Cuadrante seleccionado: CL_3_01
Área total: 4.62 km²
Geometría: Polygon


## [SC-02] Utilidades de Orientación y Rotación

Funciones para calcular la orientación principal del polígono y realizar rotaciones para alinear con los ejes.

In [31]:
from shapely.geometry import Point, Polygon

def principal_orientation_angle(poly):
    """
    Calcula el ángulo del eje principal del polígono usando minimum_rotated_rectangle
    Retorna ángulo en grados
    """
    try:
        # Usar minimum_rotated_rectangle para obtener orientación
        mbr = poly.minimum_rotated_rectangle
        coords = list(mbr.exterior.coords)
        
        # Calcular el ángulo del lado más largo
        max_length = 0
        best_angle = 0
        
        for i in range(len(coords) - 1):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            
            # Longitud del lado
            length = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if length > max_length:
                max_length = length
                # Ángulo en grados
                angle_rad = math.atan2(y2 - y1, x2 - x1)
                best_angle = math.degrees(angle_rad)
        
        return best_angle
        
    except Exception as e:
        print(f"⚠️ Error en principal_orientation_angle, usando fallback PCA: {e}")
        
        # Fallback: PCA como antes
        minx, miny, maxx, maxy = poly.bounds
        
        n_points = 1000
        points = []
        attempts = 0
        max_attempts = n_points * 10
        
        while len(points) < n_points and attempts < max_attempts:
            x = np.random.uniform(minx, maxx)
            y = np.random.uniform(miny, maxy)
            point = Point(x, y)
            if poly.contains(point):
                points.append([x, y])
            attempts += 1
        
        if len(points) < 10:
            coords = list(poly.exterior.coords)
            points = [[x, y] for x, y in coords]
        
        points = np.array(points)
        
        # PCA
        centroid = np.mean(points, axis=0)
        centered_points = points - centroid
        
        cov_matrix = np.cov(centered_points.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        principal_eigenvector = eigenvectors[:, np.argmax(eigenvalues)]
        
        angle_rad = np.arctan2(principal_eigenvector[1], principal_eigenvector[0])
        angle_deg = np.degrees(angle_rad)
        
        return angle_deg

def rotate_to_axis(geom, angle_deg):
    """
    Rota la geometría para alinearla con el eje X
    """
    return affinity.rotate(geom, -angle_deg, origin='centroid')

def rotate_back(geom_rot, angle_deg):
    """
    Rota de vuelta la geometría a su orientación original
    """
    return affinity.rotate(geom_rot, angle_deg, origin='centroid')

def area_cum(poly_rot, x):
    """
    Retorna area(poly_rot ∩ R(minx,x)) donde R es un rectángulo alineado al eje X rotado
    """
    minx, miny, maxx, maxy = poly_rot.bounds
    
    # Crear rectángulo R(minx, x)
    if x <= minx:
        return 0.0
    if x >= maxx:
        return poly_rot.area
    
    # Rectángulo desde minx hasta x
    rect = Polygon([(minx, miny), (x, miny), (x, maxy), (minx, maxy)])
    
    try:
        # Intersección del polígono con el rectángulo
        intersection = poly_rot.intersection(rect)
        
        if intersection.is_empty:
            return 0.0
        elif hasattr(intersection, 'area'):
            return intersection.area
        else:
            return 0.0
            
    except Exception as e:
        print(f"⚠️ Error en area_cum: {e}")
        return 0.0

## [SC-03] Cálculo de K Objetivo y Cortes

Funciones para calcular el número de franjas objetivo y encontrar las posiciones de corte para franjas de área igual.

In [32]:
from shapely.geometry import LineString

def suggest_k_from_area(poly, target_mean_km2):
    """
    Sugiere número de franjas basado en el área objetivo
    """
    area_total_m2 = poly.area
    area_total_km2 = area_total_m2 / 1_000_000
    k = max(2, round(area_total_km2 / target_mean_km2))
    return k

def find_equal_cuts(poly_rot, K):
    """
    Encuentra posiciones de cortes verticales para K franjas de área igual
    usando búsqueda por bisección con tolerancia 1e-6 * A_total
    """
    minx, miny, maxx, maxy = poly_rot.bounds
    A_total = poly_rot.area
    target_area_per_strip = A_total / K
    tolerance = 1e-6 * A_total
    
    positions = []
    
    for i in range(1, K):  # K-1 cortes para K franjas
        # Buscar posición x donde área acumulada sea i * target_area_per_strip
        target_area = i * target_area_per_strip
        
        # Búsqueda por bisección
        left = minx
        right = maxx
        
        while right - left > 1e-3:  # precisión de 1mm
            mid = (left + right) / 2
            area_accum = area_cum(poly_rot, mid)
            
            if area_accum < target_area:
                left = mid
            else:
                right = mid
        
        cut_position = (left + right) / 2
        positions.append(cut_position)
    
    # Validación rigurosa
    validation_areas = []
    prev_x = minx
    
    for i, x in enumerate(positions + [maxx]):
        area_strip = area_cum(poly_rot, x) - area_cum(poly_rot, prev_x)
        validation_areas.append(area_strip)
        prev_x = x
    
    sumF = sum(validation_areas)
    err = abs(sumF - A_total) / A_total if A_total > 0 else 0
    
    # Log con emojis
    print(f"🔹 Validación de áreas")
    print(f"   • Área total: {A_total:,.2f} m²")
    print(f"   • Suma franjas: {sumF:,.2f} m²")
    print(f"   • Error: {err:.4%}  {'✅' if err < 0.01 else '⚠️'}")
    
    if err >= 0.01:
        print(f"⚠️ Error de área > 1%")
    
    return positions

def build_strips(poly_rot, xs):
    """
    Construye franjas Fi_rot = (poly_rot ∩ R(x_{i-1}, x_i)).buffer(0)
    Nunca devuelve ni pinta los rectángulos R
    """
    minx, miny, maxx, maxy = poly_rot.bounds
    strips = []
    
    # Añadir límites
    x_positions = [minx] + xs + [maxx]
    
    for i in range(len(x_positions) - 1):
        x_start = x_positions[i]
        x_end = x_positions[i + 1]
        
        # Crear rectángulo R(x_start, x_end)
        rect = Polygon([
            (x_start, miny), 
            (x_end, miny), 
            (x_end, maxy), 
            (x_start, maxy)
        ])
        
        try:
            # Intersección con buffer para limpiar geometría
            strip = poly_rot.intersection(rect).buffer(0)
            
            # Verificar que no esté vacío
            if not strip.is_empty and hasattr(strip, 'area') and strip.area > 0:
                strips.append(strip)
            else:
                print(f"⚠️ Franja {i} vacía o inválida")
                
        except Exception as e:
            print(f"⚠️ Error construyendo franja {i}: {e}")
    
    return strips

## [SC-04] Corte y Postproceso

Funciones para dividir el polígono usando las posiciones calculadas y aplicar restricciones de ancho mínimo.

In [33]:
def measure_min_width(poly):
    """
    Mide ancho mínimo usando minimum_rotated_rectangle (lado menor)
    """
    try:
        mbr = poly.minimum_rotated_rectangle
        coords = list(mbr.exterior.coords)
        
        # Calcular distancias entre vértices consecutivos
        distances = []
        for i in range(len(coords) - 1):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            dist = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            distances.append(dist)
        
        # Tomar las dos distancias menores (lados del rectángulo)
        distances.sort()
        min_width = min(distances[:2]) if len(distances) >= 2 else (distances[0] if distances else 0)
        return min_width
        
    except Exception as e:
        print(f"⚠️ Error midiendo ancho: {e}")
        # Fallback: usar bbox
        minx, miny, maxx, maxy = poly.bounds
        return min(maxx - minx, maxy - miny)

def enforce_min_width(strips_rot, MIN_WIDTH_M, K_original):
    """
    Asegura ancho mínimo fusionando vecinos y re-split si es necesario
    """
    if not strips_rot:
        return strips_rot
    
    print(f"🔹 Filtro ancho mínimo (≥{MIN_WIDTH_M} m):", end=" ")
    
    # Paso 1: Fusionar franjas con ancho < MIN_WIDTH_M
    result_strips = []
    fusiones = 0
    i = 0
    
    while i < len(strips_rot):
        current_strip = strips_rot[i]
        current_width = measure_min_width(current_strip)
        
        if current_width >= MIN_WIDTH_M:
            result_strips.append(current_strip)
            i += 1
        else:
            # Fusionar con vecino
            if i + 1 < len(strips_rot):
                next_strip = strips_rot[i + 1]
                try:
                    merged = ops.unary_union([current_strip, next_strip])
                    if hasattr(merged, 'geom_type') and merged.geom_type == 'Polygon':
                        result_strips.append(merged)
                        fusiones += 1
                        i += 2
                    else:
                        result_strips.append(current_strip)
                        i += 1
                except:
                    result_strips.append(current_strip)
                    i += 1
            else:
                # Última franja, fusionar con anterior si existe
                if result_strips:
                    try:
                        prev_strip = result_strips.pop()
                        merged = ops.unary_union([prev_strip, current_strip])
                        if hasattr(merged, 'geom_type') and merged.geom_type == 'Polygon':
                            result_strips.append(merged)
                            fusiones += 1
                        else:
                            result_strips.extend([prev_strip, current_strip])
                    except:
                        result_strips.append(current_strip)
                i += 1
    
    # Paso 2: Re-split si quedamos con menos franjas de las deseadas
    resplits = 0
    if len(result_strips) < K_original:
        print(f"fusiones={fusiones}, ", end="")
        needed_splits = K_original - len(result_strips)
        
        for _ in range(needed_splits):
            # Encontrar la franja más grande
            if result_strips:
                largest_idx = max(range(len(result_strips)), 
                                key=lambda i: result_strips[i].area)
                largest_strip = result_strips[largest_idx]
                
                # Re-split transversal (en Y del polígono rotado)
                minx, miny, maxx, maxy = largest_strip.bounds
                mid_y = (miny + maxy) / 2
                
                # Crear línea de corte horizontal
                cut_line = LineString([(minx - 100, mid_y), (maxx + 100, mid_y)])
                
                try:
                    split_result = split(largest_strip, cut_line)
                    if len(split_result.geoms) >= 2:
                        # Reemplazar la franja más grande por sus dos partes
                        parts = list(split_result.geoms)
                        result_strips[largest_idx] = parts[0]
                        result_strips.insert(largest_idx + 1, parts[1])
                        resplits += 1
                    else:
                        break  # No se pudo dividir más
                except:
                    break  # Error en división
            else:
                break
        
        print(f"re-split={resplits} → K={len(result_strips)}")
    else:
        print(f"fusiones={fusiones}, K={len(result_strips)}")
    
    return result_strips

def to_gdf_subs(strips_rot, angle_deg, crs=PROJ_CRS):
    """
    Convierte franjas rotadas a GeoDataFrame final en EPSG:32618
    """
    print(f"🔹 Rotando de vuelta y calculando métricas...")
    
    data = []
    
    for i, strip_rot in enumerate(strips_rot):
        # Rotar de vuelta a orientación original
        strip_final = rotate_back(strip_rot, angle_deg)
        
        # Calcular métricas
        area_m2 = strip_final.area
        area_km2 = area_m2 / 1_000_000
        perim_m = strip_final.length
        
        # Compactness Polsby-Popper: 4πA/P²
        if perim_m > 0:
            compactness_pp = (4 * math.pi * area_m2) / (perim_m ** 2)
        else:
            compactness_pp = 0
        
        # Ancho mínimo en geometría final
        min_width_m = measure_min_width(strip_final)
        
        data.append({
            'sub_id': i,
            'geometry': strip_final,
            'area_km2': area_km2,
            'perim_m': perim_m,
            'compactness_pp': compactness_pp,
            'min_width_m': min_width_m
        })
    
    gdf = gpd.GeoDataFrame(data, crs=crs)
    return gdf

## [SC-05] Métricas y Salida

Funciones para generar el GeoDataFrame final con métricas de calidad y validación de resultados.

In [34]:
def validate_and_report_v2(gdf_subs, gdf_cuadrante, k_suggested, area_range_km2=AREA_RANGE_KM2, min_width_m=MIN_WIDTH_M):
    """
    Validación completa con tests de QA obligatorios y logs con emojis
    """
    K = len(gdf_subs)
    areas = gdf_subs['area_km2'].values
    widths = gdf_subs['min_width_m'].values
    compactness = gdf_subs['compactness_pp'].values
    
    # Obtener polígono del cuadrante
    cuadrante_poly = gdf_cuadrante.geometry.iloc[0]
    if cuadrante_poly.geom_type == 'MultiPolygon':
        cuadrante_poly = ops.unary_union(cuadrante_poly)
    
    print(f"🧮 Métricas de Subcuadrantes")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    # Métricas básicas
    area_total_km2 = areas.sum()
    area_cuadrante_km2 = cuadrante_poly.area / 1_000_000
    area_mean = areas.mean()
    area_min = areas.min()
    area_max = areas.max()
    area_std = areas.std()
    coef_var = area_std / area_mean if area_mean > 0 else 0
    
    print(f"📊 K: {K} | K sugerido: {k_suggested}")
    print(f"📏 Área total subs: {area_total_km2:.3f} km² | Cuadrante: {area_cuadrante_km2:.3f} km²")
    print(f"📈 Media: {area_mean:.3f} km² | Min: {area_min:.3f} | Max: {area_max:.3f}")
    print(f"📉 Desv.std: {area_std:.3f} | CoefVar: {coef_var:.3f}")
    
    # Porcentaje en rango
    in_range = ((areas >= area_range_km2[0]) & (areas <= area_range_km2[1])).sum()
    pct_in_range = (in_range / K) * 100
    
    # Cumplimiento de ancho mínimo
    width_compliant = (widths >= min_width_m).sum()
    pct_width_compliant = (width_compliant / K) * 100
    
    # Compacidad media
    compactness_mean = compactness.mean()
    
    print(f"🎯 En rango [{area_range_km2[0]}-{area_range_km2[1]}] km²: {in_range}/{K} ({pct_in_range:.1f}%)")
    print(f"🔧 Compacidad media: {compactness_mean:.3f}")
    print(f"📐 Ancho ≥ {min_width_m}m: {width_compliant}/{K} ({pct_width_compliant:.1f}%)")
    
    print(f"\n🧪 Tests de QA Obligatorios")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    qa_passed = True
    
    # Test 1: Todos los subcuadrantes están dentro del cuadrante
    buffer_cuadrante = cuadrante_poly.buffer(1e-6)
    within_tests = []
    for idx, row in gdf_subs.iterrows():
        is_within = row.geometry.within(buffer_cuadrante)
        within_tests.append(is_within)
        if not is_within:
            print(f"   ⚠️ Sub-ID {row['sub_id']} NO está dentro del cuadrante")
    
    all_within = all(within_tests)
    print(f"🔍 Geometrías dentro: {'✅' if all_within else '❌'} ({sum(within_tests)}/{len(within_tests)})")
    if not all_within:
        qa_passed = False
    
    # Test 2: Error de áreas < 1%
    area_error = abs(area_total_km2 - area_cuadrante_km2) / area_cuadrante_km2 if area_cuadrante_km2 > 0 else 0
    area_test_passed = area_error < 0.01
    print(f"📊 Error áreas: {area_error:.4%} {'✅' if area_test_passed else '❌'} (< 1%)")
    if not area_test_passed:
        qa_passed = False
    
    # Test 3: Sin geometrías vacías
    empty_areas = (areas == 0).sum()
    no_empty_areas = empty_areas == 0
    print(f"🚫 Áreas vacías: {'✅' if no_empty_areas else '❌'} ({empty_areas} encontradas)")
    if not no_empty_areas:
        qa_passed = False
    
    # Test 4: Áreas > 0 m²
    min_area_m2 = (areas * 1_000_000).min()
    positive_areas = min_area_m2 > 0
    print(f"➕ Área mín > 0: {'✅' if positive_areas else '❌'} ({min_area_m2:.2f} m²)")
    if not positive_areas:
        qa_passed = False
    
    print(f"\n✅ Criterios de Aceptación")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    # Criterios de aceptación
    k_match = (K == k_suggested)
    area_criteria = (pct_in_range >= 80)
    compactness_criteria = (compactness_mean >= 0.25)
    width_criteria = (pct_width_compliant == 100)
    
    print(f"🎯 K == sugerido: {'✅' if k_match else '❌'} ({K} vs {k_suggested})")
    print(f"📏 ≥80% en rango área: {'✅' if area_criteria else '❌'} ({pct_in_range:.1f}%)")
    print(f"🔧 Compactness ≥ 0.25: {'✅' if compactness_criteria else '❌'} ({compactness_mean:.3f})")
    print(f"📐 Ancho mínimo: {'✅' if width_criteria else '❌'} ({pct_width_compliant:.1f}%)")
    
    # Resultado general
    all_criteria = k_match and area_criteria and compactness_criteria and width_criteria
    overall_passed = all_criteria and qa_passed
    
    print(f"\n🏆 RESULTADO: {'✅ ACEPTADO' if overall_passed else '❌ REQUIERE AJUSTES'}")
    if not qa_passed:
        print(f"⚠️  Tests de QA fallaron - revisar geometrías")
    
    return {
        'accepted': overall_passed,
        'qa_passed': qa_passed,
        'K': K,
        'k_suggested': k_suggested,
        'area_stats': {
            'total_subs': area_total_km2,
            'total_cuadrante': area_cuadrante_km2,
            'error_pct': area_error,
            'mean': area_mean,
            'min': area_min,
            'max': area_max,
            'std': area_std,
            'coef_var': coef_var,
            'pct_in_range': pct_in_range
        },
        'compactness_mean': compactness_mean,
        'width_compliance': pct_width_compliant,
        'criteria': {
            'k_match': k_match,
            'area_range': area_criteria,
            'compactness': compactness_criteria,
            'min_width': width_criteria,
            'all_within': all_within,
            'area_error_ok': area_test_passed,
            'no_empty_areas': no_empty_areas,
            'positive_areas': positive_areas
        }
    }

## [SC-06] Flujo Completo - Equal Area Strips

Demostración del algoritmo completo de subdivisión por franjas de área igual.

In [35]:
# FLUJO COMPLETO - EQUAL AREA STRIPS V2 (ACTUALIZADO)
print("🧭 Iniciando subdivisión por Equal Area Strips (versión corregida)...")
print("=" * 60)

# 1. Calcular orientación principal
print("🔹 1) Calculando orientación principal...")
angle_deg = principal_orientation_angle(geom)
print(f"   Ángulo principal: {angle_deg:.2f}°")

# 2. Rotar polígono
print("🔹 2) Rotando polígono al eje principal...")
geom_rot = rotate_to_axis(geom, angle_deg)
minx_rot, miny_rot, maxx_rot, maxy_rot = geom_rot.bounds
print(f"   Dimensiones rotadas: {(maxx_rot-minx_rot):.0f}m x {(maxy_rot-miny_rot):.0f}m")

# 3. Calcular K objetivo
print("🔹 3) Calculando número de franjas...")
k_suggested = suggest_k_from_area(geom, TARGET_MEAN_AREA_KM2)
print(f"   K sugerido: {k_suggested} | A_target: {TARGET_MEAN_AREA_KM2:.2f} km²")

# 4. Encontrar posiciones de corte (usar nueva función)
print("🔹 4) Encontrando posiciones de corte...")
try:
    cut_positions = find_equal_cuts(geom_rot, k_suggested)
    print(f"   Encontradas {len(cut_positions)} posiciones de corte")
except Exception as e:
    print(f"   ERROR en posiciones de corte: {e}")
    cut_positions = []

if cut_positions:
    # 5. Construir franjas (usar nueva función)
    print("🔹 5) Construyendo franjas...")
    strips_rot = build_strips(geom_rot, cut_positions)
    print(f"   Generadas {len(strips_rot)} franjas iniciales")
    
    # 6. Aplicar restricción de ancho mínimo (CORREGIDO: añadir K_original)
    print("🔹 6) Aplicando restricción de ancho mínimo...")
    strips_rot_filtered = enforce_min_width(strips_rot, MIN_WIDTH_M, k_suggested)
    
    # 7. Convertir a GeoDataFrame final (usar nueva función)
    print("🔹 7) Generando GeoDataFrame final...")
    gdf_subcuadrantes = to_gdf_subs(strips_rot_filtered, angle_deg, crs=PROJ_CRS)
    
    # 8. Validar y reportar resultados (usar nueva función)
    print("🧮 8) Métricas y validación...")
    validation_result = validate_and_report_v2(gdf_subcuadrantes, cuadrante_sel, k_suggested, AREA_RANGE_KM2, MIN_WIDTH_M)
    
    # 9. Generar mapa (usar nueva función)
    print("🗺️  9) Generando mapa...")
    try:
        mapa_demo = folium_subquadrantes_map_v2(cuadrante_sel, gdf_subcuadrantes, OUTPUT_HTML)
        display(mapa_demo)
        print(f"   ✅ Mapa guardado exitosamente")
    except Exception as e:
        print(f"   ❌ Error generando mapa: {e}")
    
    print(f"\n🎯 Flujo completado con {len(gdf_subcuadrantes)} subcuadrantes")
    
else:
    print("❌ ERROR: No se pudieron calcular las posiciones de corte")
    gdf_subcuadrantes = None
    validation_result = None

🧭 Iniciando subdivisión por Equal Area Strips (versión corregida)...
🔹 1) Calculando orientación principal...
   Ángulo principal: -29.94°
🔹 2) Rotando polígono al eje principal...
   Dimensiones rotadas: 3907m x 2616m
🔹 3) Calculando número de franjas...
   K sugerido: 7 | A_target: 0.70 km²
🔹 4) Encontrando posiciones de corte...
🔹 Validación de áreas
   • Área total: 4,615,162.61 m²
   • Suma franjas: 4,615,162.61 m²
   • Error: 0.0000%  ✅
   Encontradas 6 posiciones de corte
🔹 5) Construyendo franjas...
   Generadas 7 franjas iniciales
🔹 6) Aplicando restricción de ancho mínimo...
🔹 Filtro ancho mínimo (≥120.0 m): fusiones=0, K=7
🔹 7) Generando GeoDataFrame final...
🔹 Rotando de vuelta y calculando métricas...
🧮 8) Métricas y validación...
🧮 Métricas de Subcuadrantes
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 K: 7 | K sugerido: 7
📏 Área total subs: 4.615 km² | Cuadrante: 4.615 km²
📈 Media: 0.659 km² | Min: 0.659 | Max: 0.659
📉 Desv.std: 0.000 | CoefVar: 0.000
🎯 En rango [0.5-1.1] km²: 7/7 (1

   ✅ Mapa guardado exitosamente

🎯 Flujo completado con 7 subcuadrantes


## [SC-07] Mapa Folium

Función para generar mapa interactivo con el cuadrante y sus subcuadrantes usando Folium.

In [24]:
import random
from IPython.display import display

def folium_subquadrantes_map_v2(gdf_cuadrante, gdf_subs, output_html):
    """
    Genera mapa Folium SOLO con cuadrante y subcuadrantes finales
    NO incluye rectángulos de corte ni líneas auxiliares
    """
    print(f"🗺️  Generando mapa interactivo...")
    
    # Convertir a EPSG:4326 para Folium
    gdf_cuad_wgs84 = gdf_cuadrante.to_crs('EPSG:4326')
    gdf_subs_wgs84 = gdf_subs.to_crs('EPSG:4326')
    
    # Obtener bounds del cuadrante para centrar el mapa
    bounds = gdf_cuad_wgs84.total_bounds  # [minx, miny, maxx, maxy]
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    # Crear mapa base
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=14,
        tiles='OpenStreetMap'
    )
    
    # Capa 1: SOLO el polígono del cuadrante (borde gris grueso)
    for idx, row in gdf_cuad_wgs84.iterrows():
        codigo = row.get('codigo', 'N/A')
        folium.GeoJson(
            row.geometry,
            style_function=lambda x: {
                'fillColor': 'transparent',
                'color': '#444444',
                'weight': 3,
                'fillOpacity': 0,
                'opacity': 1.0
            },
            popup=folium.Popup(f"<b>Cuadrante:</b> {codigo}", parse_html=True),
            tooltip=f"Cuadrante: {codigo}"
        ).add_to(m)
    
    # Paleta de colores fijos para subcuadrantes
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#54A0FF', '#5F27CD']
    
    # Capa 2: SOLO subcuadrantes finales (sin rectángulos auxiliares)
    for idx, row in gdf_subs_wgs84.iterrows():
        color = colors[idx % len(colors)]
        
        # Información completa para popup
        popup_html = f"""
        <div style='font-family: Arial; font-size: 12px; max-width: 200px;'>
        <b>🏷️ Subcuadrante:</b> {row['sub_id']}<br>
        <b>📏 Área:</b> {row['area_km2']:.4f} km²<br>
        <b>🔧 Compacidad:</b> {row['compactness_pp']:.3f}<br>
        <b>📐 Ancho mín:</b> {row['min_width_m']:.1f} m<br>
        <b>📏 Perímetro:</b> {row['perim_m']:.0f} m
        </div>
        """
        
        tooltip_text = f"Sub-{row['sub_id']}: {row['area_km2']:.3f} km²"
        
        folium.GeoJson(
            row.geometry,
            style_function=lambda x, color=color: {
                'fillColor': color,
                'color': '#333333',
                'weight': 1.5,
                'fillOpacity': 0.3,
                'opacity': 0.8
            },
            popup=folium.Popup(popup_html, parse_html=True, max_width=250),
            tooltip=tooltip_text
        ).add_to(m)
    
    # Ajustar vista exacta a los bounds del cuadrante
    southwest = [bounds[1], bounds[0]]  # [min_lat, min_lon]
    northeast = [bounds[3], bounds[2]]  # [max_lat, max_lon]
    m.fit_bounds([southwest, northeast])
    
    # Guardar HTML
    m.save(output_html)
    print(f"   💾 Guardado en: {output_html}")
    
    return m

## [SC-08] Función Principal Encapsulada

Función que encapsula todo el proceso Equal Area Strips para facilitar su uso.

In [25]:
def equal_area_strips_pipeline_v2(cuadrante_poly, target_area_km2=TARGET_MEAN_AREA_KM2, min_width_m=MIN_WIDTH_M, crs=PROJ_CRS):
    """
    Pipeline completo Equal Area Strips con logging de emojis exacto
    """
    
    print(f"🧭 Iniciando subdivisión por Equal-Area Strips")
    
    # 1. Calcular ángulo principal
    angle_deg = principal_orientation_angle(cuadrante_poly)
    print(f"🔹 1) Ángulo principal: {angle_deg:.2f}°")
    
    # 2. Rotar polígono y mostrar dimensiones
    poly_rot = rotate_to_axis(cuadrante_poly, angle_deg)
    minx, miny, maxx, maxy = poly_rot.bounds
    width_m = maxx - minx
    height_m = maxy - miny
    print(f"🔹 2) Dimensiones rotadas: {width_m:.0f} m × {height_m:.0f} m")
    
    # 3. Calcular K sugerido
    k_suggested = suggest_k_from_area(cuadrante_poly, target_area_km2)
    print(f"🔹 3) K sugerido: {k_suggested} | A_target: {target_area_km2:.2f} km²")
    
    # 4. Encontrar cortes con validación
    print(f"🔹 4) Cortes encontrados: ", end="")
    cut_positions = find_equal_cuts(poly_rot, k_suggested)
    print(f"{len(cut_positions)}")
    
    # 5. Construir franjas
    strips_rot = build_strips(poly_rot, cut_positions)
    print(f"🔹 5) Franjas iniciales: {len(strips_rot)}")
    
    # 6. Aplicar filtro de ancho mínimo con resplit
    strips_rot_filtered = enforce_min_width(strips_rot, min_width_m, k_suggested)
    
    # 7. Convertir a GeoDataFrame final
    gdf_subs = to_gdf_subs(strips_rot_filtered, angle_deg, crs)
    print(f"🔹 7) Subcuadrantes finales: {len(gdf_subs)}")
    
    return gdf_subs, k_suggested

def run_complete_pipeline(codigo_cuadrante="CL_3_01"):
    """
    Ejecuta pipeline completo con el logging exacto especificado
    """
    
    try:
        # Cargar datos
        gdf_cuads = load_cuadrante(CUADRANTES_PATH, PROJ_CRS)
        cuadrante_gdf = gdf_cuads[gdf_cuads['codigo'] == codigo_cuadrante]
        
        if len(cuadrante_gdf) == 0:
            print(f"❌ Error: No se encontró cuadrante {codigo_cuadrante}")
            return None, None, None
            
        # Obtener polígono
        cuadrante_poly = cuadrante_gdf.geometry.iloc[0]
        if cuadrante_poly.geom_type == 'MultiPolygon':
            cuadrante_poly = ops.unary_union(cuadrante_poly)
        
        # Ejecutar pipeline
        gdf_subs, k_suggested = equal_area_strips_pipeline_v2(cuadrante_poly)
        
        # 8. Métricas y validación
        print(f"🧮 8) Métricas...")
        print(f"🧪 9) Validación...")
        validation = validate_and_report_v2(gdf_subs, cuadrante_gdf, k_suggested)
        
        # Mostrar resumen de criterios
        crit = validation['criteria']
        k_icon = '✅' if crit['k_match'] else '❌'
        area_icon = '✅' if crit['area_range'] else '❌'
        comp_icon = '✅' if crit['compactness'] else '❌'
        width_icon = '✅' if crit['min_width'] else '❌'
        
        print(f"{k_icon} K==sugerido | {area_icon} ≥80% en rango | {comp_icon} Compactness | {width_icon} Ancho mínimo")
        
        # 10. Generar mapa
        print(f"🗺️  10) ", end="")
        mapa = folium_subquadrantes_map_v2(cuadrante_gdf, gdf_subs, OUTPUT_HTML)
        
        return gdf_subs, validation, mapa
        
    except Exception as e:
        print(f"❌ Error en pipeline: {e}")
        return None, None, None

## [SC-09] Orquestación Completa

Demostración del flujo completo con parametrización y alternativas en caso de no cumplir criterios.

In [26]:
# ========================================
# PIPELINE EXACTO - EQUAL AREA STRIPS V2
# ========================================

# Ejecutar pipeline completo con logging exacto
print("=" * 60)
print("EJECUTANDO PIPELINE CORREGIDO")
print("=" * 60)

# Código del cuadrante a procesar
codigo_objetivo = "CL_3_01"

# Ejecutar pipeline completo
gdf_subs_final, validation_final, mapa_final = run_complete_pipeline(codigo_objetivo)

if gdf_subs_final is not None:
    # Mostrar mapa
    display(mapa_final)
    
    print(f"\n🎯 Pipeline completado exitosamente")
    print(f"📁 Resultados guardados en: {OUTPUT_HTML}")
    
else:
    print(f"❌ Error: No se pudo completar el pipeline")

EJECUTANDO PIPELINE CORREGIDO
🧭 Iniciando subdivisión por Equal-Area Strips
🔹 1) Ángulo principal: -29.94°
🔹 2) Dimensiones rotadas: 3907 m × 2616 m
🔹 3) K sugerido: 7 | A_target: 0.70 km²
🔹 4) Cortes encontrados: 🔹 Validación de áreas
   • Área total: 4,615,162.61 m²
   • Suma franjas: 4,615,162.61 m²
   • Error: 0.0000%  ✅
6
🔹 5) Franjas iniciales: 7
🔹 Filtro ancho mínimo (≥120.0 m): fusiones=0, K=7
🔹 Rotando de vuelta y calculando métricas...
🔹 7) Subcuadrantes finales: 7
🧮 8) Métricas...
🧪 9) Validación...
🧮 Métricas de Subcuadrantes
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 K: 7 | K sugerido: 7
📏 Área total subs: 4.615 km² | Cuadrante: 4.615 km²
📈 Media: 0.659 km² | Min: 0.659 | Max: 0.659
📉 Desv.std: 0.000 | CoefVar: 0.000
🎯 En rango [0.5-1.1] km²: 7/7 (100.0%)
🔧 Compacidad media: 0.394
📐 Ancho ≥ 120.0m: 7/7 (100.0%)

🧪 Tests de QA Obligatorios
━━━━━━━━━━━━━━━━━━━━━━━━━━
   ⚠️ Sub-ID 0 NO está dentro del cuadrante
   ⚠️ Sub-ID 1 NO está dentro del cuadrante
   ⚠️ Sub-ID 2 NO está dentro d


🎯 Pipeline completado exitosamente
📁 Resultados guardados en: pruebas/subcuadrantes_equal_area.html


## [SC-10] Demo Rápido con Datos Actuales

Generación rápida de mapa con los datos ya procesados en el notebook.

In [ ]:
# DEMO: Probar con diferentes cuadrantes
print("🧪 DEMO: Probando pipeline con diferentes cuadrantes")
print("=" * 55)

# Lista de cuadrantes para probar
cuadrantes_test = ["CL_3_01", "CL_2_02", "CL_1_01"]

for i, codigo in enumerate(cuadrantes_test, 1):
    print(f"\n🔬 Test {i}: Cuadrante {codigo}")
    print("-" * 30)
    
    try:
        # Verificar si existe el cuadrante
        gdf_test = load_cuadrante(CUADRANTES_PATH, PROJ_CRS)
        if codigo not in gdf_test['codigo'].values:
            print(f"⚠️  Cuadrante {codigo} no encontrado, saltando...")
            continue
            
        # Ejecutar pipeline rápido (sin mapa)
        cuadrante_gdf = gdf_test[gdf_test['codigo'] == codigo]
        cuadrante_poly = cuadrante_gdf.geometry.iloc[0]
        if cuadrante_poly.geom_type == 'MultiPolygon':
            cuadrante_poly = ops.unary_union(cuadrante_poly)
        
        # Pipeline básico
        area_km2 = cuadrante_poly.area / 1_000_000
        k_sugerido = suggest_k_from_area(cuadrante_poly, TARGET_MEAN_AREA_KM2)
        
        print(f"📏 Área: {area_km2:.3f} km² → K sugerido: {k_sugerido}")
        
        # Solo ejecutar completo si es el cuadrante principal
        if codigo == "CL_3_01":
            print(f"🎯 Ejecutando pipeline completo...")
            gdf_result, validation, mapa = run_complete_pipeline(codigo)
            
            if validation and validation['accepted']:
                print(f"✅ Pipeline exitoso para {codigo}")
            else:
                print(f"⚠️  Pipeline con ajustes necesarios para {codigo}")
        else:
            print(f"ℹ️  Pipeline básico OK para {codigo}")
            
    except Exception as e:
        print(f"❌ Error procesando {codigo}: {e}")

print(f"\n🏁 Demo completado")

Generando mapa con datos actuales...
Mapa guardado en: pruebas/subcuadrantes_equal_area.html
✓ Mapa generado exitosamente
✓ Guardado en: pruebas/subcuadrantes_equal_area.html

RESUMEN DEL MAPA:
- Cuadrante: CL_3_01
- Subcuadrantes: 5
- CRS original: EPSG:32618
